In [1]:
df=spark.read.option("header","true").csv("/bank/bank_full_processed1.csv")

In [2]:
import math
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import when
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, PolynomialExpansion
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.ml.functions import vector_to_array

In [ ]:
numerical_cols = [
    'age_normalized', 'balance_normalized', 'duration', 'campaign', 'pdays', 'previous'
]
categorical_cols = ['job', 'marital', 'education', 'poutcome', 'housing', 'loan', 'month']
#use age_normalized, balance_normalized, duration, campaign, pdays, previous for PolynomialExpansion like
#age_normalized* balance_normalized and duration^2
#use job, marital, education, poutcome, housing, loan, month for OneHotEncoder

In [4]:
for col_name in numerical_cols:
    df = df.withColumn(col_name, df[col_name].cast(DoubleType()))

In [5]:
indexers = [StringIndexer(inputCol=col, outputCol=col + "_index", handleInvalid="keep") for col in categorical_cols]
encoders = [OneHotEncoder(inputCol=col + "_index", outputCol=col + "_vec") for col in categorical_cols]

In [6]:
numerical_assembler = VectorAssembler(inputCols=numerical_cols, outputCol="numerical_features")

In [7]:
poly_expansion = PolynomialExpansion(degree=2, inputCol="numerical_features", outputCol="poly_features")

In [8]:
label_indexer = StringIndexer(inputCol='y', outputCol='label')

In [9]:
assembler_inputs = ["poly_features"] + [col + "_vec" for col in categorical_cols]
final_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

In [10]:
pipeline = Pipeline(stages=[numerical_assembler, poly_expansion] + indexers + encoders + [label_indexer, final_assembler])


In [11]:
transformed_df = pipeline.fit(df).transform(df)
model_data = transformed_df.select("features", "label")

In [12]:
balance_ratio = model_data.filter("label = 0.0").count() / model_data.filter("label = 1.0").count()
weighted_data = model_data.withColumn("weight", when(model_data.label == 1.0, balance_ratio).otherwise(1.0))

In [13]:
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=2505153)
print(train_data.count())
print(test_data.count())

36184


9027


In [14]:
lsvc = LinearSVC(weightCol="weight")
paramGrid = ParamGridBuilder().addGrid(lsvc.regParam, [0.1]).build()
evaluator = BinaryClassificationEvaluator(metricName="areaUnderROC")
cv = CrossValidator(estimator=lsvc,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=5)
cv_model = cv.fit(weighted_data)

In [15]:
predictions = cv_model.transform(model_data)

In [16]:
predictions.crosstab("label", "prediction").show()

+----------------+-----+----+
|label_prediction|  0.0| 1.0|
+----------------+-----+----+
|             1.0| 1058|4231|
|             0.0|33985|5937|
+----------------+-----+----+



In [17]:
multi_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
accuracy = multi_evaluator.setMetricName("accuracy").evaluate(predictions)
precision = multi_evaluator.setMetricName("weightedPrecision").evaluate(predictions)
f1 = multi_evaluator.setMetricName("f1").evaluate(predictions)

In [18]:
TP = predictions.filter("label = 1.0 AND prediction = 1.0").count()
TN = predictions.filter("label = 0.0 AND prediction = 0.0").count()
FP = predictions.filter("label = 0.0 AND prediction = 1.0").count()
FN = predictions.filter("label = 1.0 AND prediction = 0.0").count()
mcc_numerator = (TP * TN) - (FP * FN)
mcc_denominator_sq = (TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)
mcc = 0.0 if mcc_denominator_sq == 0 else mcc_numerator / math.sqrt(mcc_denominator_sq)
auc = cv_model.avgMetrics[0]

In [19]:
print(f"Accuracy   : {accuracy:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"F1 Score   : {f1:.4f}")
print(f"AUC (Avg CV): {auc:.4f}")
print(f"MCC        : {mcc:.4f}")

Accuracy   : 0.8453
Precision  : 0.9050
F1 Score   : 0.8647
AUC (Avg CV): 0.9045
MCC        : 0.5013


In [21]:
exit()